# Phase 2 — Data Cleaning & Preparation
**Project:** Customer Churn & Retention Analytics Platform  
**Target:** Data Analyst / Business Intelligence Portfolio  
**Dataset:** IBM Telco Customer Churn (7,043 Records)  

---

### Objectives
1. Load and inspect the raw customer churn dataset.
2. Check for duplicate records and validate primary key uniqueness.
3. Identify and impute missing/blank values in `TotalCharges`.
4. Validate correct data types across numeric and categorical variables.
5. Engineer business-aligned derived variables (Tenure Buckets, Revenue Tiers, Service Counts, Risk Scores, and 2x2 Strategic Quadrant).
6. Export the reliable analytical dataset to `data/processed/customers_cleaned.csv`.


In [1]:
import os
import pandas as pd
import numpy as np

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
print('Libraries loaded successfully.')


Libraries loaded successfully.


## 1. Load Raw Dataset & Inspect Dimensions

In [2]:
raw_path = '../data/raw/customer_churn.csv'
df_raw = pd.read_csv(raw_path)
print(f'Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
df_raw.head()


Raw dataset shape: 7043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Check column types and non-null values
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 2. Check for Duplicate Records & Verify Primary Key

In [4]:
dup_count = df_raw.duplicated(subset=['customerID']).sum()
print(f'Number of duplicate customerID entries: {dup_count}')
assert dup_count == 0, 'Duplicate records detected!'


Number of duplicate customerID entries: 0


## 3. Handle Blank Values in TotalCharges
Notice that `TotalCharges` is parsed as `object` (string) because 11 records have blank space `' '`.
All 11 records have `tenure == 0` (brand-new customers signed up in current cycle).
We impute these to `0.0` and cast the column to `float64`.


In [5]:
blank_mask = df_raw['TotalCharges'].str.strip() == ''
print(f'Number of blank TotalCharges records: {blank_mask.sum()}')
df_raw[blank_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]


Number of blank TotalCharges records: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [6]:
df = df_raw.copy()
df['TotalCharges'] = df['TotalCharges'].str.strip().replace('', '0.0')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0.0)

print(f'TotalCharges dtype: {df["TotalCharges"].dtype}')
print(f'Null count in TotalCharges: {df["TotalCharges"].isna().sum()}')


TotalCharges dtype: float64
Null count in TotalCharges: 0


## 4. Feature Engineering & Derived Variables
According to the PRD specifications, we construct:
- `SeniorCitizenLabel`: 'Yes' / 'No'
- `ChurnFlag`: 1 (Churned) / 0 (Retained)
- `TenureGroup`: '0-12 Months', '13-24 Months', '25-48 Months', '49+ Months'
- `TenureCohort`: 'New (0-1 yr)', 'Growing (1-2 yrs)', 'Established (2-4 yrs)', 'Loyal (4+ yrs)'
- `MonthlyRevenue`: Alias of MonthlyCharges
- `AnnualRevenue`: MonthlyCharges * 12
- `CustomerValueSegment`: Low Value (<$35), Medium Value ($35-$70), High Value (>$70)
- `ContractCategory`: Standard contract type
- `PaymentCategory`: Automatic vs Manual
- `ServiceCount`: Count of subscribed services across 8 products
- `ChurnRiskSegment`: Rule-based risk tier (High, Medium, Low Risk)
- `StrategicSegment`: 2x2 Value & Risk Quadrant


In [7]:
# 1. Senior Citizen Label
df['SeniorCitizenLabel'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

# 2. Binary Churn Flag
df['ChurnFlag'] = (df['Churn'] == 'Yes').astype(int)

# 3. Tenure Group
def get_tenure_group(tenure):
    if tenure <= 12:
        return '0-12 Months'
    elif tenure <= 24:
        return '13-24 Months'
    elif tenure <= 48:
        return '25-48 Months'
    else:
        return '49+ Months'

df['TenureGroup'] = df['tenure'].apply(get_tenure_group)

# 4. Tenure Cohort
def get_tenure_cohort(tenure):
    if tenure <= 12:
        return 'New (0-1 yr)'
    elif tenure <= 24:
        return 'Growing (1-2 yrs)'
    elif tenure <= 48:
        return 'Established (2-4 yrs)'
    else:
        return 'Loyal (4+ yrs)'

df['TenureCohort'] = df['tenure'].apply(get_tenure_cohort)

# 5. Revenue Metrics
df['MonthlyRevenue'] = df['MonthlyCharges']
df['AnnualRevenue'] = (df['MonthlyCharges'] * 12).round(2)

# 6. Customer Value Segment
def get_revenue_segment(monthly):
    if monthly < 35.0:
        return 'Low Value'
    elif monthly <= 70.0:
        return 'Medium Value'
    else:
        return 'High Value'

df['CustomerValueSegment'] = df['MonthlyCharges'].apply(get_revenue_segment)

# 7. Contract & Payment Categories
df['ContractCategory'] = df['Contract']
df['PaymentCategory'] = df['PaymentMethod'].apply(lambda pm: 'Automatic' if 'automatic' in pm.lower() else 'Manual')

# 8. Service Count
service_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df['ServiceCount'] = df[service_cols].apply(lambda row: sum(val == 'Yes' for val in row), axis=1)

# Service Specific Flags
df['HasInternet'] = (df['InternetService'] != 'No').astype(int)
df['HasTechSupport'] = (df['TechSupport'] == 'Yes').astype(int)
df['HasOnlineSecurity'] = (df['OnlineSecurity'] == 'Yes').astype(int)

# 9. Churn Risk Scoring
def calculate_risk_score(row):
    score = 0
    if row['Contract'] == 'Month-to-month':
        score += 2
    if row['InternetService'] == 'Fiber optic' and row['TechSupport'] != 'Yes':
        score += 2
    elif row['InternetService'] == 'Fiber optic':
        score += 1
    if row['tenure'] <= 12:
        score += 1
    if row['PaymentMethod'] == 'Electronic check':
        score += 1
    return score

df['RiskScore'] = df.apply(calculate_risk_score, axis=1)

def get_risk_tier(score):
    if score >= 4:
        return 'High Risk'
    elif score >= 2:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['ChurnRiskSegment'] = df['RiskScore'].apply(get_risk_tier)

# 10. Strategic Segment (2x2 Matrix)
def get_strategic_segment(row):
    is_high_val = row['CustomerValueSegment'] == 'High Value'
    risk = row['ChurnRiskSegment']
    if is_high_val and risk == 'High Risk':
        return 'High Value + High Risk (Retention Priority)'
    elif is_high_val and risk in ['Low Risk', 'Medium Risk']:
        return 'High Value + Low/Med Risk (Loyal Champions)'
    elif not is_high_val and risk == 'High Risk':
        return 'Low/Med Value + High Risk (At-Risk Core)'
    else:
        return 'Low/Med Value + Low/Med Risk (Stable Core)'

df['StrategicSegment'] = df.apply(get_strategic_segment, axis=1)
print(f'Feature engineering complete. Total columns: {df.shape[1]}')


Feature engineering complete. Total columns: 37


## 5. Cleaned Dataset Validation & Persistence

In [8]:
# Quality Assertions
assert df.isna().sum().sum() == 0, 'Cleaned dataset contains null values!'
assert len(df) == 7043, 'Cleaned dataset row count unexpected!'

output_file = '../data/processed/customers_cleaned.csv'
os.makedirs(os.path.dirname(output_file), exist_ok=True)
df.to_csv(output_file, index=False)
print(f'Successfully persisted cleaned dataset to: {output_file}')
df.head()


Successfully persisted cleaned dataset to: ../data/processed/customers_cleaned.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,SeniorCitizenLabel,ChurnFlag,TenureGroup,TenureCohort,MonthlyRevenue,AnnualRevenue,CustomerValueSegment,ContractCategory,PaymentCategory,ServiceCount,HasInternet,HasTechSupport,HasOnlineSecurity,RiskScore,ChurnRiskSegment,StrategicSegment
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,No,0,0-12 Months,New (0-1 yr),29.85,358.20,Low Value,Month-to-month,Manual,1,1,0,0,4,High Risk,Low/Med Value + High Risk (At-Risk Core)
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,No,0,25-48 Months,Established (2-4 yrs),56.95,683.40,Medium Value,One year,Manual,3,1,0,1,0,Low Risk,Low/Med Value + Low/Med Risk (Stable Core)
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,No,1,0-12 Months,New (0-1 yr),53.85,646.20,Medium Value,Month-to-month,Manual,3,1,0,1,3,Medium Risk,Low/Med Value + Low/Med Risk (Stable Core)
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,No,0,25-48 Months,Established (2-4 yrs),42.30,507.60,Medium Value,One year,Automatic,3,1,1,1,0,Low Risk,Low/Med Value + Low/Med Risk (Stable Core)
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,No,1,0-12 Months,New (0-1 yr),70.70,848.40,High Value,Month-to-month,Manual,1,1,0,0,6,High Risk,High Value + High Risk (Retention Priority)
